# Boston Climate Modeler — Deep Learning Exploration

This notebook walks through:
1. **EDA** — time series patterns, seasonality, and variable correlations
2. **LSTM training** — stacked LSTM with pure-TF training loop; loss curves
3. **Transformer training** — encoder with multi-head attention; attention heatmaps
4. **Model comparison** — Seasonal Naive vs Ridge vs LSTM vs Transformer

**Requirements:** `pip install tensorflow numpy matplotlib jupyterlab`  
**Run from the project root:** `jupyter lab notebooks/climate_exploration.ipynb`

In [ ]:
import sys, os
# Make the package importable when running from notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # suppress TF info/warning logs

import numpy as np
import tensorflow as tf
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import date

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow {tf.__version__}  |  NumPy {np.__version__}')

In [ ]:
from climate_modeling.data import load_station_records, parse_iso_date
from climate_modeling.sequence_dataset import (
    build_sequence_dataset, SEQ_TARGET_NAMES, N_FEATURES, SEQUENCE_FEATURE_NAMES
)
from climate_modeling.tf_models import LSTMForecaster, TransformerForecaster
from climate_modeling.tf_trainer import train_model, evaluate_model

DATA_PATH = os.path.join(PROJECT_ROOT, '962598.csv')
STATION   = 'READING MA US'

records = load_station_records(DATA_PATH, STATION)
print(f'Loaded {len(records):,} records from {records[0].date} to {records[-1].date}')

## 1. Exploratory Data Analysis

In [ ]:
from datetime import datetime as dt

dates_all  = [r.date for r in records]
tobs_all   = [r.values['TOBS'] for r in records]
prcp_all   = [r.values['PRCP'] for r in records]
snow_all   = [r.values['SNOW'] for r in records]

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
fig.suptitle('READING MA US — Full Station Record', fontsize=14, fontweight='bold')

axes[0].plot(dates_all, tobs_all, lw=0.6, color='#e63946')
axes[0].set_ylabel('Temp (°F)', fontsize=9)
axes[0].set_title('Observed Temperature (TOBS)')

axes[1].bar(dates_all, prcp_all, width=1, color='#457b9d', alpha=0.7)
axes[1].set_ylabel('Inches', fontsize=9)
axes[1].set_title('Daily Precipitation (PRCP)')

axes[2].bar(dates_all, snow_all, width=1, color='#a8dadc', alpha=0.8)
axes[2].set_ylabel('Inches', fontsize=9)
axes[2].set_title('Daily Snowfall (SNOW)')

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# Seasonal decomposition: average temperature by day-of-year
from collections import defaultdict

doy_tobs  = defaultdict(list)
doy_prcp  = defaultdict(list)
for r in records:
    doy = r.date.timetuple().tm_yday
    doy_tobs[doy].append(r.values['TOBS'])
    doy_prcp[doy].append(r.values['PRCP'])

doys      = sorted(doy_tobs.keys())
mean_tobs = [np.mean(doy_tobs[d]) for d in doys]
mean_prcp = [np.mean(doy_prcp[d]) for d in doys]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(doys, mean_tobs, color='#e63946', lw=2)
axes[0].fill_between(doys, mean_tobs, alpha=0.15, color='#e63946')
axes[0].set(title='Mean Temperature by Day-of-Year', xlabel='Day of Year', ylabel='°F')

axes[1].bar(doys, mean_prcp, width=1, color='#457b9d', alpha=0.75)
axes[1].set(title='Mean Precipitation by Day-of-Year', xlabel='Day of Year', ylabel='Inches')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap across the 9 sequence features
from climate_modeling.sequence_dataset import _build_daily_features, _build_daily_targets

feats = _build_daily_features(records, records[0].date)   # (n, 9)
names = list(SEQUENCE_FEATURE_NAMES)

corr = np.corrcoef(feats.T)  # (9, 9)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(names, fontsize=8)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', fontsize=6,
                color='white' if abs(corr[i, j]) > 0.6 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Feature Correlation Matrix', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Build Sequence Dataset

Extended date range uses the full available history:  
**Train: 2012–2016** (~1 760 windows) · **Test: 2017** (~305 windows)

In [ ]:
LOOKBACK = 60   # days of history used as model input
HORIZON  = 7    # days ahead to forecast

X_train, y_train, X_test, y_test, scaler = build_sequence_dataset(
    records,
    train_start=date(2012, 1, 1),
    train_end=date(2016, 12, 31),
    test_start=date(2017, 1, 1),
    test_end=date(2017, 12, 31),
    lookback=LOOKBACK,
    horizon=HORIZON,
)

print(f'Training windows : {len(X_train):>5}  shape={X_train.shape}')
print(f'Test     windows : {len(X_test):>5}  shape={X_test.shape}')
print(f'\nTarget normalisation (original scale):')
for i, t in enumerate(SEQ_TARGET_NAMES):
    print(f'  {t}  mean={scaler.target_means[i]:.3f}  std={scaler.target_stds[i]:.3f}')

## 3. LSTM Forecaster

Stacked 2-layer LSTM implemented entirely with `tf.Module` and `tf.Variable` — **no `tf.keras`**.

In [ ]:
lstm = LSTMForecaster(
    input_size=N_FEATURES,
    hidden_size=64,
    n_layers=2,
    horizon=HORIZON,
    dropout_rate=0.3,
    name='lstm_forecaster',
)

n_params = sum(int(tf.size(v)) for v in lstm.trainable_variables)
print(f'LSTM trainable parameters: {n_params:,}')
for v in lstm.trainable_variables:
    print(f'  {v.name:<40} {str(v.shape)}')

In [ ]:
print('Training LSTM …')
history_lstm, _ = train_model(
    lstm, X_train, y_train,
    epochs=150, batch_size=32, lr=1e-3, patience=15, verbose=True,
)
print('Done.')

In [ ]:
# Training loss curves
fig, ax = plt.subplots(figsize=(9, 4))
epochs = range(1, len(history_lstm['train_loss']) + 1)
ax.plot(epochs, history_lstm['train_loss'], label='Train', color='#2563eb', lw=2)
ax.plot(epochs, history_lstm['val_loss'],   label='Validation', color='#dc2626', lw=2, linestyle='--')
ax.set(title='LSTM — Training & Validation Loss (MSE, normalised scale)',
       xlabel='Epoch', ylabel='MSE')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
metrics_lstm, pred_lstm = evaluate_model(lstm, X_test, y_test, scaler)

print('LSTM — test-set metrics (all 7 forecast days combined):')
print(f"  {'Target':<6}  {'RMSE':>8}  {'MAE':>8}  {'R²':>8}")
for t, m in metrics_lstm.items():
    print(f"  {t:<6}  {m['rmse']:>8.3f}  {m['mae']:>8.3f}  {m['r2']:>8.3f}")

In [ ]:
# 7-day forecast examples: show first 6 test windows
true_vals  = scaler.inverse_transform_targets(y_test)
n_examples = 6

fig, axes = plt.subplots(n_examples, 3, figsize=(13, 3 * n_examples))
fig.suptitle('LSTM — 7-Day Forecast Examples (test set)', fontsize=13, fontweight='bold')

for row in range(n_examples):
    for col, target in enumerate(SEQ_TARGET_NAMES):
        ax = axes[row, col]
        ax.plot(range(1, HORIZON + 1), true_vals[row, :, col], 'o-', label='Actual', color='#1d3557', lw=2)
        ax.plot(range(1, HORIZON + 1), pred_lstm[row, :, col], 's--', label='LSTM',   color='#e63946', lw=2)
        if row == 0:
            ax.set_title(target, fontsize=11, fontweight='bold')
        ax.set_xlabel('Day ahead')
        if col == 0:
            ax.set_ylabel(f'Window {row+1}')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Transformer Forecaster

Encoder-only Transformer with sinusoidal positional encoding and multi-head self-attention — all built from raw `tf.Variable` and `tf.linalg` ops.

In [ ]:
transformer = TransformerForecaster(
    input_size=N_FEATURES,
    d_model=32,
    n_heads=2,
    n_layers=3,
    d_ff=64,
    horizon=HORIZON,
    dropout_rate=0.3,
    name='transformer_forecaster',
)

n_params = sum(int(tf.size(v)) for v in transformer.trainable_variables)
print(f'Transformer trainable parameters: {n_params:,}')
for v in transformer.trainable_variables:
    print(f'  {v.name:<50} {str(v.shape)}')

In [ ]:
print('Training Transformer …')
history_tf, _ = train_model(
    transformer, X_train, y_train,
    epochs=150, batch_size=32, lr=1e-3, patience=15, verbose=True,
)
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
epochs_tf = range(1, len(history_tf['train_loss']) + 1)
ax.plot(epochs_tf, history_tf['train_loss'], label='Train',      color='#7c3aed', lw=2)
ax.plot(epochs_tf, history_tf['val_loss'],   label='Validation', color='#d97706', lw=2, linestyle='--')
ax.set(title='Transformer — Training & Validation Loss (MSE, normalised scale)',
       xlabel='Epoch', ylabel='MSE')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
metrics_transformer, pred_transformer = evaluate_model(transformer, X_test, y_test, scaler)

print('Transformer — test-set metrics (all 7 forecast days combined):')
print(f"  {'Target':<6}  {'RMSE':>8}  {'MAE':>8}  {'R²':>8}")
for t, m in metrics_transformer.items():
    print(f"  {t:<6}  {m['rmse']:>8.3f}  {m['mae']:>8.3f}  {m['r2']:>8.3f}")

In [ ]:
# Attention heatmap — last encoder block, first test sample
sample_x = tf.constant(X_test[:1], dtype=tf.float32)   # (1, lookback, N_FEATURES)
_, attn_weights = transformer(sample_x, training=False)

# Plot the last encoder block's first head
last_block_attn = attn_weights[-1].numpy()[0]  # (n_heads, lookback, lookback)
n_heads = last_block_attn.shape[0]

fig, axes = plt.subplots(1, n_heads, figsize=(5 * n_heads, 4))
if n_heads == 1:
    axes = [axes]
fig.suptitle('Transformer — Attention Heatmaps (last block, 1st test window)', fontsize=12)

for h, ax in enumerate(axes):
    im = ax.imshow(last_block_attn[h], cmap='Blues', aspect='auto')
    ax.set_title(f'Head {h+1}')
    ax.set_xlabel('Key position (day)')
    ax.set_ylabel('Query position (day)')
    plt.colorbar(im, ax=ax, shrink=0.7)

plt.tight_layout()
plt.show()

## 5. All-Model Comparison

| Model | PRCP RMSE | PRCP R² | SNOW RMSE | SNOW R² | TOBS RMSE | TOBS R² |
|---|---|---|---|---|---|---|
| Seasonal Naive | 0.304 | — | 0.934 | — | 10.661 | — |
| Ridge Regression | 0.259 | −0.025 | 0.717 | −0.034 | 8.171 | 0.747 |
| **LSTM** (this notebook) | see below | | | | | |
| **Transformer** (this notebook) | see below | | | | | |

In [ ]:
print('\n=== Full model comparison ===')
print(f"{'Model':<18}  ", end='')
for t in SEQ_TARGET_NAMES:
    print(f'{t} RMSE  {t} R²     ', end='')
print()
print('-' * 80)

def print_row(name, metrics):
    print(f'{name:<18}  ', end='')
    for t in SEQ_TARGET_NAMES:
        m = metrics[t]
        print(f"{m['rmse']:>8.3f}  {m['r2']:>7.3f}  ", end='')
    print()

print_row('LSTM',        metrics_lstm)
print_row('Transformer', metrics_transformer)

# Visualise TOBS (best-performing target) comparison
tobs_idx = list(SEQ_TARGET_NAMES).index('TOBS')
n_plot   = min(100, len(true_vals))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(n_plot), true_vals[:n_plot, 0, tobs_idx], label='Actual', color='#1d3557', lw=1.5)
ax.plot(range(n_plot), pred_lstm[:n_plot, 0, tobs_idx],        label='LSTM (day+1)',        color='#e63946', lw=1.5, alpha=0.8)
ax.plot(range(n_plot), pred_transformer[:n_plot, 0, tobs_idx], label='Transformer (day+1)', color='#7c3aed', lw=1.5, alpha=0.8, linestyle='--')
ax.set(title='TOBS — Day+1 Forecast vs Actual (first 100 test windows)',
       xlabel='Test window index', ylabel='Temperature (°F)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()